# Заргат — генерация 3D-модели через Hunyuan3D-2mini

Подготовленный запуск для Google Colab. Эта версия сначала создаёт **геометрию без текстуры**: так выше шанс уложиться в доступную бесплатную видеопамять. Риггинг и игровые анимации будут отдельным этапом.

В Colab выбери GPU в меню среды выполнения. Бесплатный GPU выдаётся не всегда. Не покупай подписку ради этого теста.

In [ ]:
import os, torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('GPU не выдана. Открой Среда выполнения → Сменить среду выполнения → GPU. Если GPU недоступна бесплатно, остановись здесь.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 1. Установка официального Hunyuan3D-2

Клонируем исходный код Tencent непосредственно из официального репозитория.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
# Сохраняем установленный в Colab PyTorch/CUDA: исключаем torch и torchvision из общего списка.
!grep -v -E '^(torch|torchvision)$' requirements.txt > /tmp/hy3d-requirements.txt
!pip install -r /tmp/hy3d-requirements.txt
!pip install -e .
print('Установка зависимостей завершена. Текстурные CUDA-модули не собираем: на первом проходе нужна только форма.')

## 2. Загрузи подготовленный фронтальный кадр Заргата

Используй файл `zargat_front_input.jpg`, который подготовлен отдельно из утверждённого концепта. В Colab нажми кнопку загрузки и выбери именно этот файл. Нужен один персонаж целиком, не лист с несколькими ракурсами.

In [ ]:
from google.colab import files
from PIL import Image
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Изображение не загружено')
input_image_path = '/content/' + next(iter(uploaded.keys()))
source_image = Image.open(input_image_path).convert('RGB')
display(source_image.resize((512, 512)))

## 3. Генерация формы (Hunyuan3D-2mini)

Используется компактная модель 0.6B. Она создаёт меш по одному изображению, но не гарантирует точное совпадение всех деталей и не создаёт скелет.

In [ ]:
import torch
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mini',
    subfolder='hunyuan3d-dit-v2-mini',
    use_safetensors=True,
    device='cuda'
)
with torch.inference_mode():
    result = shape_pipe(
        image=input_image_path,
        num_inference_steps=30,
        octree_resolution=380,
        num_chunks=20000,
        generator=torch.manual_seed(12345),
        output_type='trimesh'
    )
mesh = result[0]
output_path = '/content/zargat_shape.glb'
mesh.export(output_path)
print('Создано:', output_path)
print('Размер MB:', round(os.path.getsize(output_path) / 1024**2, 2))

Скачай GLB и передай его сюда. Это пока **не готовый игровой персонаж**: мы отдельно проверим геометрию, силуэт, крылья и оружие, затем подготовим материалы, скелет и анимации. Не отправляем модель в APK до твоего просмотра и одобрения.

In [ ]:
files.download(output_path)